# Week 8 · Thursday: Semantic search and embedding pipelines

Tuesday and Wednesday stored 36 help-centre articles that someone wrote by hand, each one tidy. Today 1,000 real forum posts go through the whole pipeline a team runs before anything is searchable: load, clean, tag, describe, embed, check, write, search. First into Chroma, then the same records into Pinecone. Today:

1. **Semantic search:** a search by meaning finds what a keyword search misses, and where keywords still win
2. **Vector embedding models:** what the model is, two models compared, and how to choose one
3. **Data preparation:** loading 2,365 forum posts and choosing what to keep
4. **Data preprocessing:** cleaning, dropping short and duplicate posts, and cutting long ones
5. **Named entity recognition (NER):** finding the organisations, people and places each post names
6. **Metadata processing:** the fields a search filters on and shows, in a shape both databases accept
7. **Vector embedding in batches:** 1,000 posts through the model, 64 at a time
8. **Preparing for upsert:** records, a check before writing, and a batched upsert into Chroma
9. **Vector query with semantic search:** a search function with filters, and whether it is any good
10. **The same in Pinecone:** the same records in your own Pinecone index, filtered on lists, then cleaned up

Sections 3 to 9 are one pipeline: each section reads what the one before it made, so run them in order.

Parts marked *Going deeper* are optional on a first pass: run their cells, then come back to them with the notes page.

**Studying on your own**

This notebook is written to be worked through alone as well as followed live. Allow about three and a quarter hours for all of it. The ends of sections 2, 5 and 9 are good places to stop for a break.

| Sections | What you work through | About |
|---|---|---|
| Setup, 1 and 2 | keyword against meaning, and the embedding model opened up | 45 min |
| 3 to 5 | loading, cleaning and tagging the 1,000 posts | 45 min |
| 6 and 7 | metadata, and embedding in batches | 35 min |
| 8 and 9 | records into Chroma, and a search function with filters | 35 min |
| 10 | the same records in Pinecone, then the clean-up | 25 min |
| Recap | the Quick check | 10 min |

How to work a subsection:

- Read the text above the code cell and answer its **Predict first** question before you run anything. Write the answer down: a guess you have committed to is what makes the result stick.
- Run the cell. Compare what it prints with your prediction and with the bullets under it. Counts and scores quoted in the text come from our own run and should match yours; seconds depend on the machine, so yours will differ.
- Do the **Try changing** edit, see what moved, then put the value back and rerun the cell. Later cells rely on the original values.
- A **Common mistake** bullet names an error people often make at that step.
- **Work it by hand first** marks a short paper exercise, in 2.2. Do it before you run the cell.
- A line such as "Check your answer" opens when you click it. Try first, then open it.
- Each section ends with **Before you move on**. If you cannot do one of its items, reread that subsection before going on.
- The Quick check at the end has its answers folded under it. Say each answer aloud before you open them: these are the questions an interviewer asks.
- If the runtime disconnects, click the cell you were on and choose Runtime > Run before. Everything above it runs again, the installs included (about a minute), and 8.3 empties the Chroma folder before refilling it, so nothing collides.
- Section 10 needs a free Pinecone account with its API key saved in Colab's Secrets panel (the key icon on the left) as `PINECONE_API_KEY`, with "Notebook access" switched on. Without one, Runtime > Run all runs sections 1 to 9 and stops at the first cell of section 10. If you stop partway through section 10, run 10.3 before you leave, so your Pinecone project is left empty.
- The [notes page for today](https://2494-gen-ai-cn.github.io/training/weeks/week-08/day-4-semantic-search-and-embeddings.html) covers the same ideas in prose, goes further in places, and ends with a short check.

Run the next cell first. It installs the week's two pinned libraries, Chroma and the Pinecone client (about a minute on a fresh runtime), imports what we use all day, defines three small helpers and fixes the random seed. If Colab asks you to restart the session after the installs, choose Runtime > Restart session and run the cell again.

In [ ]:
# ---------- Setup and plotting helpers (run once; no need to read live) ----------
try:
    import google.colab                      # present only on Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:                                 # the week's two pinned installs (a local run uses its own copies)
    %pip install -q chromadb==1.5.9
    %pip install -q pinecone==9.1.0

import json
import os
import re
import shutil
import sys
import time
from collections import Counter

import numpy as np
import pandas as pd
import sklearn
import spacy
import chromadb
import pinecone
import sentence_transformers
from pinecone import Pinecone, ServerlessSpec
from sentence_transformers import SentenceTransformer
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 64)


def chroma_hits(result):
    """A Chroma query result as a table: id, distance, then topic, orgs and the start of the preview."""
    return pd.DataFrame([
        {"id": i, "distance": round(d, 3), "topic": m["topic"], "orgs": m.get("orgs", []),
         "preview": m["preview"][:60]}
        for i, d, m in zip(result["ids"][0], result["distances"][0], result["metadatas"][0])])


def pinecone_hits(result):
    """A Pinecone query result as a table: id, score, then topic, orgs and the start of the preview."""
    return pd.DataFrame([
        {"id": m.id, "score": round(m.score, 3), "topic": m.metadata["topic"],
         "orgs": m.metadata.get("orgs", []), "preview": m.metadata["preview"][:60]}
        for m in result.matches])


def wait_for_count(index, expected, namespace=None):
    """Poll describe_index_stats every 2 s, for up to 60 s, until the index (or one namespace) holds `expected` vectors."""
    for _ in range(31):
        stats = index.describe_index_stats()
        if namespace is None:
            count = stats.total_vector_count
        else:
            count = stats.namespaces[namespace].vector_count if namespace in stats.namespaces else 0
        if count >= expected:
            break
        time.sleep(2)
    print(f"{count} vectors" + ("" if namespace is None else f" in namespace {namespace!r}"))
    return count


print(f"Python {sys.version.split()[0]}, numpy {np.__version__}, pandas {pd.__version__}, "
      f"scikit-learn {sklearn.__version__}, sentence-transformers {sentence_transformers.__version__}, "
      f"spaCy {spacy.__version__}, chromadb {chromadb.__version__}, pinecone {pinecone.__version__}")
print("Setup complete")

## 1. Semantic search

Every search box answers one question: which stored texts best match what was typed? There are two ways to decide what "match" means, and this section puts them side by side on eight short sentences before any real data arrives.

### 1.1 Keywords against meaning

**Keyword search** (also called lexical search) ranks stored texts by the words they share with the query. Keyword engines can only find documents based on word matches.

**Semantic search** ranks stored texts by how close their meaning is to the query's, by comparing embeddings: the lists of numbers a model computes for each text. A match needs no shared word.

First the model, the same two lines Tuesday used as a black box: `encoder` loads `all-MiniLM-L6-v2` (downloaded on first use), and `embed` turns a list of texts into one row of 384 numbers per text, each row scaled to length 1. Section 2 opens it up.

In [ ]:
start = time.perf_counter()
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")   # downloads on first use


def embed(texts):
    return encoder.encode(texts, normalize_embeddings=True).astype("float32")


print(f"MiniLM loaded in {time.perf_counter() - start:.1f} s; embed(['hello']) has shape {embed(['hello']).shape}")

Now eight invented sentences, `SENTENCES`, and one query: `"my car will not start when it is freezing"`. Only the first sentence, about an engine on cold mornings, is about that problem, and it shares no content word with the query.

The keyword search is week 6's `TfidfVectorizer`. `fit_transform` learns a vocabulary from the eight sentences and weights each word; `transform` scores the query with that same vocabulary, so a query word that no sentence uses counts for nothing. `cosine_similarity` then compares the query's word scores with each sentence's. By default the vectorizer lowercases, keeps words of two or more letters or digits and uses no stop-word list, so "when", "it" and "is" count like any other word.

**Predict first:** which sentence will TF-IDF rank first for the car query, and what will the engine sentence score?

In [ ]:
SENTENCES = [
    "The engine struggles to turn over on cold mornings.",
    "My phone battery dies before lunch.",
    "The dentist said I need a filling.",
    "Rocket launches are delayed when the wind is too strong.",
    "This recipe needs two cups of flour.",
    "When it is sunny, the park fills with families.",
    "I will not pay that parking ticket.",
    "The printer shows error code E-104 after every page.",
]
query = "my car will not start when it is freezing"          # Try changing it: see the bullets below

tfidf = TfidfVectorizer()                                     # learns its vocabulary from the 8 sentences
keyword_scores = cosine_similarity(tfidf.fit_transform(SENTENCES), tfidf.transform([query]))[:, 0]
for i in np.argsort(-keyword_scores)[:3]:
    print(f"keyword  {keyword_scores[i]:.2f}  {SENTENCES[i]}")
print(f"engine sentence: {keyword_scores[0]:.2f}")

- "car", "start" and "freezing" appear in no sentence, so TF-IDF has only "my", "when it is" and "will not" to go on. The sunny-park sentence comes first at about 0.37 and the parking-ticket sentence second at about 0.35. Both are wrong.
- The engine sentence, the only right answer, scores exactly 0.00: no shared word, no score.

Now the same query by meaning. Every sentence and the query become 384 numbers, and `@` takes a dot product: multiply two lists position by position and add up. Because `embed` scales every vector to length 1, that sum is the cosine similarity, where 1 means the same direction and 0 means unrelated.

**Predict first:** will the embedding search put the engine sentence first, and will its score be near 1?

In [ ]:
meaning_scores = embed(SENTENCES) @ embed([query])[0]     # length-1 vectors: the dot product is the cosine similarity
for i in np.argsort(-meaning_scores)[:3]:
    print(f"meaning  {meaning_scores[i]:.2f}  {SENTENCES[i]}")

- The engine sentence comes first at about 0.36, well ahead of the next at about 0.22. MiniLM was trained on sentence pairs that mean the same thing, so "freezing" lands near "cold mornings" and "car will not start" near "engine struggles to turn over".
- 0.36 is not a percentage. What counts is the order and the gap to the next row. A short query against a longer sentence is what the Sentence Transformers docs call asymmetric search, and scores there rarely come near 1.
- Try changing it: in the cell with `SENTENCES`, set `query = "my phone will not start when it is freezing"`, which shares "my phone" with the battery sentence, and rerun both cells. TF-IDF still puts the sunny-park sentence first (about 0.34), with the battery and parking sentences just behind (about 0.32 each). MiniLM moves the battery sentence to first (about 0.39) and the engine sentence to second (about 0.23): it follows the meaning, now about a phone. Put the car query back and rerun both cells before 1.2.
- **Common mistake:** comparing the two columns of numbers with each other. A TF-IDF score of 0.37 and a cosine similarity of 0.36 come from different scales. Compare the order within each column only.

**In an interview:** keyword search ranks by shared words, so a paraphrase with no shared word scores zero. Semantic search compares embeddings from a model trained on pairs of texts that mean the same, so "my car will not start when it is freezing" finds "the engine struggles to turn over on cold mornings".

### 1.2 Where keywords win

Lexical search suits specific terms such as product codes, error strings and names; semantic search suits an answer phrased in different words. Here are four sentences, three of them about filter parts with codes made of the same digits, and a query that is just one code, `"7731-B"`.

**Predict first:** which part will MiniLM rank first for the query `"7731-B"`, and which will TF-IDF?

In [ ]:
PARTS = [
    "Replacement filter part 7731-B fits the 2019 model.",
    "Replacement filter part 7713-B fits the 2021 model.",
    "Replacement filter part 1377-B fits the 2017 model.",
    "Filters should be changed every six months.",
]
part_query = "7731-B"
tfidf_parts = TfidfVectorizer()
part_keyword = cosine_similarity(tfidf_parts.fit_transform(PARTS), tfidf_parts.transform([part_query]))[:, 0]
part_meaning = embed(PARTS) @ embed([part_query])[0]
print(pd.DataFrame({"sentence": PARTS, "keyword (TF-IDF)": part_keyword.round(2),
                    "meaning (MiniLM)": part_meaning.round(2)}))

- TF-IDF scores only the exact part, at about 0.47, and gives every other sentence 0. It treats "7731" as its own word, so only the sentence holding it matches.
- MiniLM ranks the wrong part, 7713-B, first at about 0.23, just above the right one at about 0.21. To a model trained on meaning, two codes made of the same digits mean nearly the same thing.
- **Hybrid search** combines two or more retrieval signals, a lexical one (full-text **BM25**, another word-scoring formula, or sparse vectors) and a dense one (embeddings), because each method misses what the others catch. We name it here and do not build it.
- Try changing it: in the 1.1 cell, set `query = "E-104"` and rerun it and the cell under it. Both searches put the printer sentence first (TF-IDF about 0.35, MiniLM about 0.38), because it is the only code in that set. Codes trip the model when several similar codes sit side by side, as here. Put the car query back and rerun both cells.
- **Common mistake:** concluding that semantic search is always better, or that keyword search is out of date. The part-number result is the reason teams run both.

**In an interview:** keyword search wins on exact strings such as part numbers, error codes and rare names, where an embedding model blurs codes that look alike. Hybrid search runs both and merges the rankings, so exact codes and paraphrases are both found.

### 1.3 The pipeline

Eight hand-written sentences needed no preparation. A thousand real forum posts do. Every later section of this notebook is one step of this picture, in this order:

```
load -> prepare -> clean -> tag entities -> build metadata -> embed in batches -> build + check records -> upsert -> query
3.1     3.2        4        5               6                 7                  8.1, 8.2                 8.3      9, then 10 in Pinecone
```

- **Load** the posts and read a few. **Prepare**: choose the topics and how many posts to keep.
- **Clean**: whitespace, empty, short and duplicate posts, and long ones cut to a known length.
- **Tag entities**: the organisations, people and places each post names. **Build metadata**: the fields a search filters on and shows.
- **Embed in batches**: one vector per post. **Build records**, check them, and **upsert** them in batches.
- **Query**: a search function with filters, and a check that it is any good.

Retrieval for a language model's answer reuses exactly this pipeline; week 9 adds the model on top.

**Predict first:** which two steps of this pipeline will take the longest to run?

<details>
<summary>Check your answer</summary>

Tagging entities (section 5) and embedding (section 7): both run a model over every post. That is why the notebook starts from a sample of 1,000 posts, small enough to rerun from the top while you change things.

</details>

- **Common mistake:** thinking the vector database cleans or tags the text. It stores and compares whatever it is given. Everything before the upsert is the team's work.

**In an interview** ("What has to happen before the first search works?"): load and look at the data, clean it, drop what should not be returned, tag the entities a user will filter on, build small metadata, embed every text with one model in batches, check the records, upsert them in batches, then query with the same model.

**Before you move on**, check that you can:

- name a query that keyword search misses and semantic search finds, and say why the keyword score is exactly 0;
- name two kinds of string where keyword search beats semantic search;
- say what the dot product of two length-1 vectors equals;
- list the pipeline's steps in order.

## 2. Vector embedding models

The embedding model produces every vector; the database only stores and compares them. So the model decides what "close" means. This section opens `encoder` up.

### 2.1 What the model is

A **sentence transformer** turns a whole sentence or short passage into one fixed-length vector. `all-MiniLM-L6-v2` is one, licensed Apache-2.0. It was trained by contrastive learning on over 1 billion sentence pairs: given one sentence, pick its true partner out of a random set. Texts that mean the same end up with vectors that point the same way.

Three properties a team reads before using a model, because each one sets something in the database:

- **Output size**, `encoder.get_embedding_dimension()`: the numbers per vector, which a Pinecone index needs as its `dimension`.
- **Length limit**, `encoder.max_seq_length`: the most **word pieces** (the sub-word units the model reads; a long or rare word can be several) it reads from one text. Longer input is cut off, with no error.
- **Normalisation**: `normalize_embeddings=True` scales each vector to length 1, so a dot product is the cosine similarity and the database's metric is cosine. The default of `encode` is `False`, which is why `embed` passes `True`.

**Predict first:** what will `np.linalg.norm` print for one embedded sentence, and what shape will that one sentence have?

In [ ]:
print(f"output size {encoder.get_embedding_dimension()}, length limit {encoder.max_seq_length} word pieces")

one = embed(["The engine struggles to turn over on cold mornings."])   # Try changing it: a three-word sentence
print(one.shape, one.dtype, f"length {np.linalg.norm(one[0]):.3f}")
print("first five numbers:", one[0, :5].round(3))

- 384 numbers per vector and a limit of 256 word pieces. A Pinecone index for this model needs `dimension=384`.
- One sentence gives shape `(1, 384)` with dtype `float32`: one row per text, one column per number, 4 bytes per number. `encode` always returns that two-dimensional shape for a list of texts.
- The length prints 1.000 because `embed` passes `normalize_embeddings=True`.
- Chroma has another route: `SentenceTransformerEmbeddingFunction` from `chromadb.utils.embedding_functions` lets Chroma run the model itself (its `model_name` defaults to `"all-MiniLM-L6-v2"`, its `normalize_embeddings` to `False`). We keep `embedding_function=None` as Tuesday did and pass vectors from `embed`, so Chroma and Pinecone hold identical vectors.
- Try changing it: replace the sentence with a three-word one such as `"Cold engine trouble."` and rerun. The shape stays `(1, 384)`: the vector has a fixed length whatever the text's length. Put the engine sentence back.
- **Common mistake:** typing the old method name `get_sentence_embedding_dimension()`. On Colab's sentence-transformers 5.7.0 it still works but prints a `FutureWarning` saying it was renamed to `get_embedding_dimension()`.

**In an interview:** an embedding model turns each text into a fixed-length vector; the vector database stores and compares those vectors, so the model decides what counts as similar. Its output size sets the index dimension, its length limit sets what is read, and normalised vectors mean cosine is the metric.

### 2.2 Two models compared

`all-mpnet-base-v2` is a bigger sentence transformer: 768 numbers per vector, and it reads up to 384 word pieces. The Sentence Transformers docs say it provides the best quality of their general models, while MiniLM is 5 times faster and still offers good quality. The next cell downloads it on first use, which takes a while.

**Work it by hand first.** A float32 number takes 4 bytes. Before you run anything, write down how many bytes one MiniLM vector takes, how many one mpnet vector takes, and roughly how many gigabytes a million vectors of each take.

<details>
<summary>Check your answer</summary>

- MiniLM: 384 × 4 = 1,536 bytes per vector, so a million vectors take 1,536,000,000 bytes, about 1.5 GB.
- mpnet: 768 × 4 = 3,072 bytes per vector, about 3.1 GB per million.
- Both are before the index adds its own overhead.

Two usual wrong answers: 384 or 768 bytes, from forgetting that each number takes 4 bytes; and "1.5 MB per million", from dividing by a million once too often (a million times 1,536 bytes is about 1.5 billion bytes).

</details>

**Predict first:** mpnet's vectors are twice as long. Will it also take about twice as long to embed the same texts, or much longer?

In [ ]:
start = time.perf_counter()
mpnet = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")     # downloads on first use
print(f"mpnet loaded in {time.perf_counter() - start:.1f} s: {mpnet.get_embedding_dimension()} numbers, "
      f"length limit {mpnet.max_seq_length} word pieces")

In [ ]:
hundred = [" ".join(SENTENCES)] * 100          # 100 texts of about 400 characters, a typical forum post's length
for name, model in [("MiniLM", encoder), ("mpnet", mpnet)]:
    start = time.perf_counter()
    model.encode(hundred, normalize_embeddings=True)
    seconds = time.perf_counter() - start
    scores = (model.encode(SENTENCES, normalize_embeddings=True)
              @ model.encode([query], normalize_embeddings=True)[0])
    top = np.argsort(-scores)[:3]
    print(f"{name}: {seconds:.1f} s per 100 texts; top 3: "
          + "; ".join(f"{scores[i]:.2f} {SENTENCES[i][:28]}" for i in top))

- On the run behind these notes, mpnet took seven to eight times as long as MiniLM for the same 100 texts (about 8 to 9 s against 1.1 s on a laptop CPU), far more than twice: it is a much bigger network, so it does far more work per word piece, not just more output. Your seconds will differ; expect a gap of several times either way.
- Both models put the engine sentence first for the car query. mpnet scores it about 0.60.
- Storage, from the exercise above: about 1.5 GB per million MiniLM vectors against about 3.1 GB for mpnet. On this data the bigger model doubles the storage and costs far more time without changing the top answer.
- **Common mistake:** reading mpnet's higher score (about 0.60 against 0.36) as proof that it is better. Scores from two models are not on the same scale. Compare what each ranks first: here both rank the engine sentence first.

**In an interview:** a bigger embedding model can rank better but costs storage (768 × 4 = 3,072 bytes per vector against 1,536) and time (several times slower here). Pick it only if the gain shows on data like yours.

### 2.3 Choosing a model

Four things to weigh: **quality** on tasks like yours, **size** (numbers per vector, so storage), **speed**, and the **languages** the model covers.

- The **model card** is the model's page on Hugging Face: it states the output size, the length limit and the training data.
- The **MTEB leaderboard**: the Sentence Transformers docs point to it for strong models. MTEB scores embedding models on retrieval, clustering, classification, reranking and similarity tasks.
- **Hosted embedding services** are the other route: an API computes the vectors and no model runs on your machine. Named only.

The one rule: **the query vector must come from the same embedding model used at upsert**, with the same normalisation. Moving to a better model means embedding every stored text again.

**Predict first:** a teammate embeds the queries with mpnet while the index holds MiniLM vectors. What happens: an error, or wrong answers?

<details>
<summary>Check your answer</summary>

An error, and a loud one: the query has 768 numbers and the index expects 384. The dangerous case is a different model with the same output size, which raises no error at all.

</details>

- **Common mistake:** assuming that no error means the model matches. Only a different size errors. A different model of the same size returns confident nonsense with no error, because its 384 numbers mean something else, and so does a query embedded without `normalize_embeddings=True` against stored vectors that had it.

**In an interview** ("You switch to a better model next month. What happens to the stored vectors?"): every stored text has to be embedded again with the new model and written back, because queries from one model compared with vectors from another give wrong answers, with no error when the sizes match.

**Before you move on**, check that you can:

- read a model's output size and length limit in code, and say what each sets in the database;
- work out the bytes per vector and per million vectors for a model of any size;
- name four things to weigh when choosing a model;
- say what breaks, loudly and silently, when upsert and query use different models.

## 3. Data preparation

### 3.1 Loading the posts

**20 Newsgroups** is a collection of posts from 20 online discussion groups, 18,846 posts in all, which scikit-learn downloads with one function. We keep four groups, `TOPICS = ["sci.space", "sci.med", "rec.autos", "comp.graphics"]`. The arguments of `fetch_20newsgroups`:

- `subset="train"`: the training part of the collection (also the default). We need posts to search, not a model to test.
- `categories=TOPICS`: only these four groups.
- `remove=("headers", "footers", "quotes")`: strips each post's e-mail headers, its footer (the signature block) and the lines quoted from earlier posts. Only the header filter follows an exact standard; the docs warn that the other two are not always correct.
- `random_state=RANDOM_STATE`: fixes the shuffle, which is on by default, so every run gets the posts in the same order.

The function returns a **bunch**, an object whose fields `data`, `target`, `target_names` and `filenames` are read with a dot. `target` holds a number per post, and `target_names` turns the number into the group's name.

**Predict first:** `TOPICS` lists `sci.space` first. Will target number 0 mean `sci.space`?

In [ ]:
TOPICS = ["sci.space", "sci.med", "rec.autos", "comp.graphics"]
start = time.perf_counter()
raw = fetch_20newsgroups(subset="train", categories=TOPICS, remove=("headers", "footers", "quotes"),
                         random_state=RANDOM_STATE)          # downloads on first use
posts = pd.DataFrame({
    "post_id": range(len(raw.data)),                         # the row's position at load
    "topic": [raw.target_names[i] for i in raw.target],      # names looked up by number, never from TOPICS
    "text": raw.data,
})
print(f"loaded in {time.perf_counter() - start:.1f} s; target_names {raw.target_names}")
print(f"{len(posts)} posts")
print(posts["topic"].value_counts().to_string())

- `target_names` comes back sorted alphabetically, `['comp.graphics', 'rec.autos', 'sci.med', 'sci.space']`, not in the order of `TOPICS`. So 0 means `comp.graphics`, and the code looks each name up with `raw.target_names[i]`.
- 2,365 posts: rec.autos 594, sci.med 594, sci.space 593, comp.graphics 584. Nearly equal, so no topic will crowd the others out of results.
- **Common mistake:** building topic names from the order of `TOPICS`. Every label is then wrong, with no error. Read names from `target_names` by number.

Before writing any cleaning code, read two posts whole. Look for runs of blank lines, leftovers of the quote and signature filters, and anything a reader would not want returned by a search.

**Predict first:** after `remove=` took out headers, signatures and quotes, will the two posts below read as clean prose?

In [ ]:
for row in [2, 10]:                                           # Try changing it: another row number
    text = posts.loc[row, "text"]
    print(f"--- post {posts.loc[row, 'post_id']} ({posts.loc[row, 'topic']}), {len(text)} characters ---")
    print(repr(text[:700]))

- `repr` shows the text with its line breaks written as `\n`, so they are visible. The first post (sci.space) is a short question followed by the writer's signature block, a magazine advert with a postal address and prices, which the footer filter missed. The second (sci.med) starts with a line break and is full of double spaces and line breaks in mid-sentence.
- None of that is what a reader wants a search to match on. Section 4 removes the whitespace; the leftover signatures come back in 5.3. Real posts also arrive near-empty after `remove=`, which the help-centre articles never did.
- Try changing it: change `[2, 10]` to two other row numbers, such as `[0, 5]`, and rerun. You see two more raw posts, often with blank runs or a leftover signature. Put `[2, 10]` back.

**In an interview** ("You are handed a public text dataset: what do you look at first?"): the counts per label, the label names and how they map to numbers, and a few whole records read by eye, before writing any cleaning code.

### 3.2 Choosing what to keep

Two decisions before cleaning:

- **A stable id.** `post_id` is the row's position at load. It is unique, and it stays the same on every run because the shuffle is fixed by `RANDOM_STATE`. Writing the same id twice overwrites a record; a new random id on each run would pile up duplicates.
- **A sample.** We keep 250 posts per topic after cleaning, 1,000 in all, and cut them in 4.3. Tagging entities and embedding are the slow steps, and 1,000 posts keep every step fast enough to rerun from the top while you change it. The trade-off: a sample may miss rare cases the full set holds, so a team runs the whole set once the pipeline is right.

The label you already have, `topic`, is metadata for free: it becomes the first filter field, and the answer key when 9.3 checks the search.

**Predict first:** the file names in `raw.filenames` are another candidate id. Are they unique across the four topics?

In [ ]:
print(f"post_id unique: {posts['post_id'].is_unique}")
print(f"unique file names: {len(set(raw.filenames))} of {len(raw.filenames)}")

- Both are unique: every `post_id` once, and 2,365 distinct file names for 2,365 posts. The notebook uses the position because it is short; the file name would also do.
- **Common mistake:** giving records a fresh id on each run. Upsert overwrites only an id it already holds, so a new id is a new record, and the store fills with duplicates.

**In an interview** ("Why start with 1,000 posts instead of all 18,846?"): entity tagging and embedding are slow, so a sample lets you rerun the whole pipeline in minutes while you get it right; then you scale up. Each record gets a stable id taken from the data, so a rerun overwrites instead of duplicating.

**Before you move on**, check that you can:

- load four groups of 20 Newsgroups with headers, footers and quotes removed, and print the count per topic;
- say why topic names come from `target_names` and what 0 means here;
- say why each record's id must be the same on every run;
- give one reason to start from a sample and one risk of doing so.

## 4. Data preprocessing

### 4.1 Cleaning text

`clean_text(s)` returns `re.sub(r"\s+", " ", s).strip()`:

- `re.sub` replaces every match of a pattern. `\s+` means one or more whitespace characters, so a blank line, a tab and three spaces all become one space.
- `.strip()` removes whitespace left at the start and the end.
- `posts["text"].map(clean_text)` runs the function on every post in the column.

**Predict first:** after cleaning, how many of the 2,365 posts will be completely empty: none, a handful, or dozens?

In [ ]:
def clean_text(s):
    return re.sub(r"\s+", " ", s).strip()       # every run of spaces, tabs and newlines becomes one space


posts["text"] = posts["text"].map(clean_text)
print(f"empty {(posts['text'] == '').sum()}, under 100 characters {(posts['text'].str.len() < 100).sum()}")

- 77 posts are empty and 220 are under 100 characters, the 77 included.
- They were not empty to begin with. `remove=` stripped their quotes and signature, and some posts were nothing else.
- **Common mistake:** judging a post's length before cleaning. Blank runs make a near-empty post look long.

**In an interview** ("How do you clean raw text before embedding it?"): read some records first, collapse whitespace and trim, then drop empty, very short and exact-duplicate texts, and cut very long ones at a word boundary so you know what was stored. Treat a dataset's own filters with care: `remove=` is a heuristic.

### 4.2 Dropping short and duplicate posts

- **Short posts.** A post under 100 characters ("Thanks!", "Me too") says too little to be worth returning, and an empty one would still get 384 numbers that a search can return. `posts["text"].str.len() >= 100` gives True or False per row, and indexing the table with it keeps the True rows.
- **Duplicates.** An exact duplicate would take two of the five result slots with the same text. `drop_duplicates(subset="text")` keeps the first of any rows whose text is identical. It runs after cleaning, so copies that differed only in whitespace are caught too.

**Predict first:** 220 posts are under 100 characters. If 3 of the rest are exact duplicates, how many posts are left?

In [ ]:
n_loaded = len(posts)
posts = posts[posts["text"].str.len() >= 100]                 # Try changing it: 200
n_long_enough = len(posts)
posts = posts.drop_duplicates(subset="text")                  # keeps the first copy of each text
print(f"loaded {n_loaded}, short dropped {n_loaded - n_long_enough}, "
      f"duplicates {n_long_enough - len(posts)}, left {len(posts)}")
print(posts["topic"].value_counts().to_string())

- 2,365 minus 220 short posts is 2,145, minus 3 exact duplicates is 2,142.
- Per topic: sci.med 558, sci.space 546, comp.graphics 524, rec.autos 514. Every topic keeps more than 500, so 250 per topic fits.
- Try changing it: set the threshold to `200`, then rerun the 3.1 load cell, the 4.1 cell and this cell, in that order (this cell changes `posts`, so it must start from the loaded table). 487 posts are now too short and 1,875 are left, instead of 2,142. Put `100` back and rerun the same three cells, because 4.3 expects the 2,142.
- **Common mistake:** dropping duplicates before cleaning. Two copies that differ only in whitespace are not identical yet, so both survive.

**In an interview** ("Why drop very short posts and exact duplicates?"): a very short post has nothing worth returning, and a duplicate pair would fill two result slots with the same text, so the user sees one answer instead of two.

### 4.3 Length

Two steps here. First the sample from 3.2: `groupby("topic").head(250)` keeps the first 250 rows of each topic, 1,000 in all. Then a look at how long the kept posts are.

**Predict first:** a median forum post is a few hundred characters. How long do you expect the longest of the 1,000 to be?

In [ ]:
posts = posts.groupby("topic").head(250).reset_index(drop=True)    # the first 250 of each topic
lengths = posts["text"].str.len()
print(f"{len(posts)} posts; median {lengths.median():.0f} characters, "
      f"{(lengths > 1000).sum()} over 1,000, the longest {lengths.max():,}")

- The median post is 482 characters, but 227 of the 1,000 are over 1,000 and the longest runs to 55,808, more than fifty pages of text in a single post.
- MiniLM reads at most 256 word pieces and drops the rest with no error, so such a post would be embedded from its opening lines only.

So we cut every post to 1,000 characters at a word boundary, and the stored text is exactly what we chose to keep. `cut(text, limit=1000)` returns a short text unchanged; otherwise it takes the first 1,000 characters, and `rsplit(" ", 1)[0]` splits once at the last space and keeps the left part, so no word is cut in half.

**Predict first:** after the cut, will the model read every post whole?

In [ ]:
def cut(text, limit=1000):
    if len(text) <= limit:
        return text
    return text[:limit].rsplit(" ", 1)[0]       # split once at the last space: no word cut in half


posts["text"] = posts["text"].map(cut)
print(len(posts), posts["text"].str.len().max() <= 1000)

pieces = [len(encoder.tokenizer(t)["input_ids"]) for t in posts["text"]]   # word pieces per post
print(f"median {np.median(pieces):.0f} word pieces; "
      f"{sum(p > encoder.max_seq_length for p in pieces)} of {len(pieces)} over {encoder.max_seq_length}")

- `1000 True`: 1,000 posts, none longer than 1,000 characters.
- Not every post is read whole. The median post is 129 word pieces, but 63 of the 1,000 are still over 256, so the model trims those further. The 1,000 characters bound what is stored; they do not promise that the model reads it all.
- Nothing warns you during embedding. Calling the tokenizer directly, as this cell does, prints one message, "Token indices sequence length is longer than the specified maximum sequence length for this model (393 > 256)"; `encode` itself truncates without a word.
- Splitting a long document into several pieces, each embedded on its own, is the usual fix. It is called **chunking**, and week 9 teaches it.
- **Common mistake:** believing a long post is embedded whole. MiniLM reads the first 256 word pieces and drops the rest silently.

**In an interview** ("A post is 55,000 characters long: what does the model read of it?"): only its first 256 word pieces; the rest is dropped without an error. So the pipeline cuts long texts at a word boundary to a known length, and a team that needs the whole document splits it into chunks.

**Before you move on**, check that you can:

- write `clean_text` and say what `\s+` matches;
- work out how many posts are left from the loaded, short and duplicate counts;
- cut a text to a limit at a word boundary;
- say what the model does with text past its length limit, and how you would find out how many texts that affects.

## 5. Named entity recognition (NER)

### 5.1 What NER finds

**Named entity recognition (NER)** is a model that finds the names of real-world things in text and labels each one with its kind. A named entity is a real-world object that is assigned a name. It sits in the pipeline before metadata because a user may want "posts that mention NASA", which is a filter, not a similarity.

- **spaCy** is a Python library for processing text. `en_core_web_sm` 3.8.0 is its small English pipeline: 12 MB, trained on a corpus called OntoNotes 5, with 18 entity labels. Colab ships both, so nothing is installed.
- `nlp = spacy.load("en_core_web_sm")` gives a function that turns a string into a `Doc`: the text split into words, with what each component found attached.
- `doc.ents` holds every entity found, as a span of the text. `ent.text` is the words, `ent.label_` its kind as a short code, and `spacy.explain(label)` turns the code into a description.

**Predict first:** in "NASA said Sally Ride trained in Houston before the shuttle flew from Florida.", will "Sally Ride" come out as one entity or two?

In [ ]:
nlp = spacy.load("en_core_web_sm")
print(nlp.pipe_names)

doc = nlp("NASA said Sally Ride trained in Houston before the shuttle flew from Florida.")   # Try changing it
for ent in doc.ents:
    print(f"{ent.text:12} {ent.label_:7} {spacy.explain(ent.label_)}")

- Four entities: NASA ORG ("Companies, agencies, institutions, etc."), Sally Ride PERSON ("People, including fictional"), Houston GPE and Florida GPE ("Countries, cities, states").
- "Sally Ride" is one entity of two words: NER finds spans, not single words.
- We keep four labels: ORG, PERSON, GPE and LOC (other locations). GPE and LOC together fill a `places` field.
- The pipeline's components print first. NER is the last, `ner`; 5.4 switches two of the others off.
- Try changing it: replace the sentence with one of your own holding a person, a company and a city, and rerun. For "My Honda Civic broke down in Ottawa, so I called Bob at CAA." the model gives Honda ORG, Civic PRODUCT and Ottawa GPE: "Honda Civic" was split into a company and a product, and Bob and CAA were missed. Put the NASA sentence back.

The model is statistical and depends on the examples it was trained on. On its own test data its NER scores precision 84.30, recall 84.36 and F 84.33, so roughly one entity in six is missed or mislabelled even on text like its training data. Forum posts are further from that text.

**Predict first:** in "I traded my old Ford Taurus for a newer model last spring.", will the model call Ford Taurus a company or a product?

In [ ]:
for text in ["I traded my old Ford Taurus for a newer model last spring.",
             "Too much MSG in Chinese food gives me a headache and raises my BP."]:
    print(text, "->", [(ent.text, ent.label_) for ent in nlp(text).ents])

- "Ford Taurus", a car model, comes out as ORG, a company.
- "MSG" (a food additive) and "BP" (blood pressure) both come out as ORG too. None of this raises an error.
- In section 5.3 the same model tags a writer's e-mail address from a signature as an organisation.
- **Common mistake:** trusting the labels because they look confident. They come with no doubt attached, and on forum text a fair share are wrong or missing.

**In an interview:** NER finds named spans, such as organisations, people and places, and labels each one. In a search index the entities become metadata a user can filter on. The model is statistical, so on text unlike its training data it mislabels, and the entities need a look before they become a filter.

### 5.2 The NER helper function

`extract_entities(texts, batch_size=50)` takes a list of texts and returns one small dict per text, `{"orgs": [...], "people": [...], "places": [...]}`, ready for the metadata step.

- `nlp.pipe(texts, batch_size=50)` processes the texts as a stream in batches, usually more efficient than one call per text. `batch_size` is the number of texts it buffers at a time. It yields one `Doc` per text in the order of the original texts, so result number 7 belongs to text number 7.
- `KEEP` maps each label worth keeping to a field. Any other label, such as Civic's PRODUCT, gets `None` from `.get` and is skipped.
- Three rules fix the shape: every dict has all three keys, even when a list is empty; a name is added once, in the order first seen; each list stops at 10, so one long post cannot carry a huge field.

**Predict first:** for the Honda sentence, what will the helper put under `people`?

In [ ]:
KEEP = {"ORG": "orgs", "PERSON": "people", "GPE": "places", "LOC": "places"}


def extract_entities(texts, batch_size=50):
    """One dict of orgs, people and places per text, each deduplicated in first-seen order and capped at 10."""
    results = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        found = {"orgs": [], "people": [], "places": []}
        for ent in doc.ents:
            key = KEEP.get(ent.label_)
            if key and ent.text not in found[key] and len(found[key]) < 10:
                found[key].append(ent.text)
        results.append(found)
    return results


for found in extract_entities([
    "NASA said Sally Ride trained in Houston before the shuttle flew from Florida.",
    "My Honda Civic broke down in Ottawa, so I called Bob at CAA.",      # Try changing it: add "" to the list
]):
    print(found)

- The NASA sentence gives `{'orgs': ['NASA'], 'people': ['Sally Ride'], 'places': ['Houston', 'Florida']}`.
- The Honda sentence gives `{'orgs': ['Honda'], 'people': [], 'places': ['Ottawa']}`: Bob was missed, and the `people` key is still there with an empty list. Section 6 decides what to do with empty lists.
- Try changing it: add an empty string `""` as a third item of the list and rerun. A third dict prints with three empty lists and no error. Take it out again.
- **Common mistake:** calling `nlp` once per post in a plain loop. It works, but `nlp.pipe` with a batch size is the way to run many texts.

**In an interview** ("How would you run NER over 1,000 texts efficiently?"): with `nlp.pipe` and a batch size, collecting the labels you need into a fixed shape per text: every key present, names deduplicated, lists capped. Switching off components NER does not need makes it faster still.

### 5.3 Entities across 1,000 posts

Now the helper over all 1,000 posts, timed. Then `Counter` counts how often each organisation appears per topic, and `most_common(4)` returns the top four with their counts. Reading this table is how you find out whether an entity field is fit to filter on.

**Predict first:** what will top the list of organisations in the `sci.med` posts?

In [ ]:
start = time.perf_counter()
ents = extract_entities(posts["text"].tolist())
ner_seconds = time.perf_counter() - start
print(f"{len(ents)} posts tagged in {ner_seconds:.1f} s")

top_orgs = {}
for topic in sorted(TOPICS):
    counts = Counter(org for e, t in zip(ents, posts["topic"]) if t == topic for org in e["orgs"])
    top_orgs[topic] = [f"{org} ({n})" for org, n in counts.most_common(4)]
print(pd.DataFrame.from_dict(top_orgs, orient="index", columns=["1st", "2nd", "3rd", "4th"]))
nasa = sum("NASA" in e["orgs"] for e, t in zip(ents, posts["topic"]) if t == "sci.space")
print(f"sci.space posts with NASA in orgs: {nasa}")

- The 1,000 posts took about 12 s on a laptop CPU; your Colab time will differ.
- `sci.space`: NASA (28) first by far, with Apollo (9) behind it. Sensible.
- `rec.autos`: car makers, Ford (12), Toyota (10), Honda (10) and Chrysler (7), as expected.
- `comp.graphics`: VGA (11), DOS (10), TIFF (8) and GIF (7), which are standards and file formats, not organisations.
- `sci.med`: an e-mail address, `geb@cadre.dsl.pitt.edu` (27), from the signature of one writer's posts that `remove=("footers",)` did not catch, then MSG (15). A team would strip e-mail addresses in cleaning before trusting this field.
- NASA is in `orgs` for 28 `sci.space` posts, which makes it a good filter example for sections 9 and 10.
- **Common mistake:** reading VGA, DOS, TIFF and GIF as companies. The model tagged them ORG, and a filter will match whatever it tagged, right or wrong.

**In an interview** ("Is an NER field usable as a filter?"): read its top values per group first. Here NASA in the space posts is sound, but the medical posts' top "organisation" is a signature address, so that field needs cleaning before it becomes a filter.

### 5.4 A faster pipeline

*Going deeper. The one idea to keep: NER needs only some of spaCy's components, and skipping the others saves time with almost the same entities.*

`nlp.pipe(..., disable=["parser", "lemmatizer"])` skips the sentence parser and the word-root finder for this call. A disabled component stays loaded but does not run; an excluded one (`spacy.load(..., exclude=[...])`) is never loaded at all. spaCy's `n_process` spreads the work over several processes, but its start-up cost is high, so we only name it.

In [ ]:
start = time.perf_counter()
fast = [[(ent.text, ent.label_) for ent in doc.ents]
        for doc in nlp.pipe(posts["text"], batch_size=50, disable=["parser", "lemmatizer"])]
fast_orgs = [list(dict.fromkeys(t for t, label in found if label == "ORG"))[:10] for found in fast]
print(f"{time.perf_counter() - start:.1f} s with the parser and lemmatizer off, against {ner_seconds:.1f} s")
print(f"same orgs for {sum(f == e['orgs'] for f, e in zip(fast_orgs, ents))} of {len(ents)} posts")

- Faster with two components switched off: about two thirds of the time or less on our runs (7.4 s against 12.0 s on the last one).
- Almost the same answers: 995 of the 1,000 posts get exactly the same organisations, and the other 5 differ slightly. A speed-up that changes a few outputs is still a change, so compare on your own data before you switch.

**Before you move on**, check that you can:

- run spaCy on a sentence and print each entity's text, label and description;
- name the four labels this notebook keeps and the field each fills;
- write a helper over `nlp.pipe` that returns the same keys for every text;
- read a table of top entities and name one that should not be used as a filter yet, and why.

## 6. Metadata processing

### 6.1 What goes in metadata

**Metadata** is the small dict stored beside each vector. Two kinds of field go in it:

- **Fields you filter on:** `topic`, and the entity lists `orgs`, `people` and `places`, so a query can ask for space posts that mention NASA.
- **Fields you show:** `preview`, the first 300 characters, and `chars`, the post's length, so a result can be displayed without a second lookup.
- **What stays out:** everything else. The full text lives with the originals and is found by id.

Both databases limit what a record may carry. Pinecone allows at most 40 KB of metadata per record. The allowed types:

| | Chroma 1.5.9 | Pinecone |
|---|---|---|
| values | strings, integers, floats, booleans | strings, numbers, booleans |
| lists | a list of one of these, all elements the same type, **never empty** | lists of strings |
| `None` | refused by `add` (see below for `upsert`) | refused ("null metadata values aren't supported") |

We tested the empty list in both, on the week's pinned versions. Chroma refuses it, as the next cell shows. Pinecone 9.1.0 accepted a record with `"people": []` and stored it (fetching it back gave `'people': []`), and refused `"people": None` with `ApiError [400] Metadata value must be a string, number, boolean or list of strings, got 'null' for field 'people'`. Because one of the two refuses an empty list, the rule for this notebook is: **a key whose list is empty is left out**, and no value is ever `None`.

**Predict first:** will Chroma accept a record whose `people` is an empty list? And one whose `topic` is `None`?

In [ ]:
scratch = chromadb.EphemeralClient().get_or_create_collection("scratch", embedding_function=None)
for n, meta in enumerate([{"topic": "sci.space", "people": ["Sally Ride"]},     # a list of strings
                          {"topic": "sci.space", "people": []},                  # an empty list
                          {"topic": None}]):                                     # a None
    try:
        scratch.add(ids=[f"try-{n}"], embeddings=embed(["test"]), metadatas=[meta])
        print(f"accepted {meta}")
        scratch.delete(ids=[f"try-{n}"])                  # leave the scratch collection empty for a rerun
    except (ValueError, TypeError) as err:
        print(f"refused  {meta}: {type(err).__name__}: {err}")

- A list of strings is accepted: Chroma 1.5.9 stores lists as they are.
- The empty list is refused with `ValueError: Expected metadata list value for key 'people' to be non-empty in add.`, and the `None` with `TypeError: argument 'metadatas': Cannot convert Python object to MetadataValue`.
- `upsert` treats a `None` differently: in our test it accepted `{"topic": None}` and stored the record with no `topic` key at all, without a word. So in Chroma a `None` can vanish instead of failing, which is worse.
- Empty lists are common in this data: many posts name no organisation, no person or no place (6.2 counts them). Writing them as `[]` would fail the Chroma upsert partway through.
- **Common mistake:** writing `"orgs": []` or a `None` for a post where nothing was found. Leave the key out instead.

**In an interview** ("What do you put in metadata, and what do you leave out?"): the fields you filter on and the few you show, small and typed; never the full text, which stays with the originals under the same id, and never a `None` or, for Chroma, an empty list.

### 6.2 make_metadata

`make_metadata(row, ents)` starts from `{"topic": row["topic"]}`, adds `orgs`, `people` and `places` only when their list is not empty, then `preview` and `chars`. The test doing the work is `if ents[key]:`: an empty list counts as false in Python, so its key is left out.

**Predict first:** for the Honda sentence, which keys will the dict have?

In [ ]:
def make_metadata(row, ents):
    """Filter fields and display fields for one post; a key whose list is empty is left out, and no value is None."""
    meta = {"topic": row["topic"]}
    for key in ("orgs", "people", "places"):
        if ents[key]:                                 # an empty list is false: leave the key out
            meta[key] = ents[key]
    meta["preview"] = row["text"][:300]
    meta["chars"] = len(row["text"])
    return meta


print(make_metadata({"topic": "rec.autos", "text": "My Honda Civic broke down in Ottawa, so I called Bob at CAA."},
                    {"orgs": ["Honda"], "people": [], "places": ["Ottawa"]}))

- `topic`, `orgs`, `places`, `preview` and `chars`. There is no `people` key, because its list was empty.

Now all 1,000 posts. `posts.to_dict("records")` turns the table into one dict per row, and `zip` pairs each row with its entities in order.

**Predict first:** of the 1,000 dicts, will more lack an `orgs` key or a `places` key?

In [ ]:
metas = [make_metadata(row, e) for row, e in zip(posts.to_dict("records"), ents)]
print(f"{len(metas)} dicts; posts with no orgs / people / places key: "
      f"{sum('orgs' not in m for m in metas)} / {sum('people' not in m for m in metas)} / "
      f"{sum('places' not in m for m in metas)}")
print(f"any None: {any(v is None for m in metas for v in m.values())}")
sizes = [len(json.dumps(m, ensure_ascii=False).encode("utf-8")) for m in metas]
print(f"metadata as JSON: largest {max(sizes)} bytes, median {np.median(sizes):.0f} bytes")

- 359 posts name no organisation, 489 no person and 688 no place, so more lack `places` than `orgs`. Each of those keys is simply left out, and no value is `None`.
- The largest dict is 870 bytes as JSON and the median 388, far inside Pinecone's 40 KB per record.
- **Common mistake:** expecting a post without an `orgs` key to show up under an organisation filter. A missing key simply does not match, which is the right answer: NER found no organisation there.

**In an interview** ("Design metadata for a post index that users filter by topic and by organisation mentioned"): a string `topic`, a list of strings `orgs` from NER, left out when empty, plus a short `preview` and `chars` to show; no `None`, and well inside the per-record limit.

### 6.3 One shape per database

The plan for this day had a helper, `to_chroma_metadata`, to join each list into one string for Chroma. We tested the pinned Chroma 1.5.9 instead and found that it stores a non-empty list of strings as it is (6.1) and filters it, so there is nothing to convert and the helper is dropped: **the same dict goes to both databases**. Only the filter spelling changes.

| "Does this post's `orgs` list hold NASA?" | spelling |
|---|---|
| Chroma | `where={"orgs": {"$contains": "NASA"}}` |
| Pinecone | `filter={"orgs": {"$in": ["NASA"]}}`, or the bare `{"orgs": "NASA"}` |

- In Chroma, `$contains` checks whether a metadata list includes a value. 9.2 runs it, and shows that `$in` against a list field matches nothing.
- In Pinecone, a filter on a list field matches when any value in the list satisfies it, so both spellings above match `["NASA", "JPL"]`. An array as a bare filter value is an HTTP 400 error, the server refusing a badly formed request.

**Predict first:** both databases hold the same `orgs` list. Will the same filter dict work in both?

<details>
<summary>Check your answer</summary>

No. Chroma's test for "this list holds the value" is `$contains`; Pinecone's is `$in` or a bare value. Chroma's `$in` on a list field matches nothing, and raises no error, so a copied filter just looks like an empty search.

</details>

- **Common mistake:** copying a filter from one database to the other. A wrong operator usually matches nothing and raises no error, so the search just looks empty.

**In an interview:** one metadata shape can serve both Chroma and Pinecone when the lists are non-empty lists of strings; what differs is the filter operator, `$contains` in Chroma and `$in` in Pinecone.

**Before you move on**, check that you can:

- split a record's metadata into fields you filter on and fields you show;
- name the three values neither database should be sent here: a `None`, an empty list, a mixed list;
- write `make_metadata` so an empty list's key is left out;
- write the "orgs holds NASA" filter for Chroma and for Pinecone.

## 7. Vector embedding in batches

### 7.1 Why batches

A **batch** is a group of texts the model processes in one pass. Embedding in batches means feeding the posts through in groups of a fixed size, for three reasons:

- **Memory:** the model holds one batch at a time, so a million texts need no more memory than 64 do.
- **Progress:** a long run shows how far it has got.
- **Failure:** if something breaks, one batch is lost, not the whole run.

`encode` does the batching itself when you pass `batch_size`. Its default is 32; we set `EMBED_BATCH = 64`. `show_progress_bar` by default shows a bar only at a detailed logging level, so we pass `True`.

**Predict first:** how many batches will the progress bar count for 1,000 posts in batches of 64?

In [ ]:
EMBED_BATCH = 64
texts = posts["text"].tolist()

start = time.perf_counter()
vectors = encoder.encode(texts, batch_size=EMBED_BATCH, normalize_embeddings=True,    # Try changing it: batch_size=8
                         show_progress_bar=True).astype("float32")
lengths = np.linalg.norm(vectors, axis=1)
print(vectors.shape, vectors.dtype, f"{time.perf_counter() - start:.1f} s; "
      f"row lengths {lengths.min():.3f} to {lengths.max():.3f}")

- The bar counts 16 batches (1,000 divided by 64, rounded up), then `(1000, 384) float32`: one row of 384 numbers per post, each of length 1, ready for cosine.
- On the run behind these notes the 1,000 posts took about half a minute on a laptop CPU; your Colab time will differ.
- Try changing it: set `batch_size=8`, rerun, and compare the time. Before you run it, guess whether eight times as many passes will take eight times as long. On our run the time hardly moved (about 33 s at 8 against 30 to 37 s at 64 over three runs), and the shape is the same: the model's work is set by the word pieces it reads, not by how they are grouped. Put `batch_size=EMBED_BATCH` back and rerun the cell, so `vectors` is the batch-64 result the next sections expect.
- **Common mistake:** believing the batch size changes the results. It changes speed and memory, not the meaning of the vectors; 7.2 checks.

**In an interview** ("Embed a million texts: how, and why in batches?"): with the same `encode` call, a batch size and a progress bar. Memory stays flat, you see how far it has got, and a failure loses one batch, not the run. The batch size changes speed and memory only.

### 7.2 The same loop by hand

The same batching written out, the way you would write it to do something with each batch as soon as it is ready (8.3 uses the same slicing to write batches). `range(0, len(texts), EMBED_BATCH)` gives 0, 64, 128 and so on, and `texts[i:i + EMBED_BATCH]` is the slice starting there. `np.vstack` stacks the small arrays into one.

`np.allclose(a, b)` asks whether two arrays are equal within a tolerance. Its default is `rtol=1e-05, atol=1e-08`; `atol=1e-5` allows every number to differ by up to 0.00001.

**Predict first:** how many slices will the loop make, and how many posts will the last one hold? Will `np.allclose` say the hand-built array matches 7.1's?

In [ ]:
starts = range(0, len(texts), EMBED_BATCH)
by_hand = np.vstack([embed(texts[i:i + EMBED_BATCH]) for i in starts])
print(by_hand.shape, f"{len(starts)} slices, the last holding {len(texts) - starts[-1]} posts")
print("allclose, default tolerance:", np.allclose(by_hand, vectors))
print("allclose, atol=1e-5:        ", np.allclose(by_hand, vectors, atol=1e-5))
print(f"largest difference {np.abs(by_hand - vectors).max():.1e}")

- 16 slices, the last holding the 40 posts left (15 × 64 = 960), stacked into `(1000, 384)`.
- At the default tolerance `np.allclose` prints `False`; with `atol=1e-5` it prints `True`. The largest difference between any two numbers is about 0.0000002.
- Why they differ at all: inside `encode` the texts are sorted by length before they are cut into batches, and each batch is padded to its longest text. The hand loop's slices of 64 each go through `embed`, which batches them again at the default 32, so the posts are grouped differently and only the last decimal places move. Search results are unaffected.
- **Common mistake:** reading `False` as a bug. The vectors match to within 0.00001; the default tolerance is simply stricter than that.

**In an interview** ("Why might the vectors from two batching schemes differ in the last decimal places?"): the model pads each batch to its longest text and `encode` sorts by length first, so different batching means different padding around each text. The effect stays far below anything that changes a ranking.

**Before you move on**, check that you can:

- give three reasons to embed in batches;
- work out the number of batches and the size of the last one for any count and batch size;
- write the batch loop by hand with slices and `np.vstack`;
- say what changing the batch size changes and what it does not.

## 8. Preparing for upsert

### 8.1 Records

A **record** is what we write: `{"id": f"post-{post_id}", "values": vec.tolist(), "metadata": meta}`, the dict Pinecone's client takes. We build one per post by zipping the rows, the vectors and `metas`. `UPSERT_BATCH = 100` is the number of records written per call in 8.3 and 10.1.

`vec.tolist()` turns a numpy row into a plain list of numbers. The pinned Pinecone 9.1.0 cannot turn a numpy array into JSON: `upsert` with a numpy array as values raises `TypeError: Type is not JSON serializable: numpy.float32` before any request is sent. Chroma takes either.

The vector database is one tool among several. In an application the originals stay in files or a SQL table keyed by the same id; the vector store holds the id, the vector and the metadata needed to filter and show; a search returns ids that lead back to the original:

```
originals (files or SQL table)   key: post-<id>   full text
      |  same id
vector store (Chroma / Pinecone): id, vector (384 numbers), metadata (topic, orgs, ..., preview, chars)
      |  search returns ids
back to the original by id
```

**Predict first:** what will the first record's id look like, and how many numbers will its `values` hold?

In [ ]:
UPSERT_BATCH = 100
records = [{"id": f"post-{row['post_id']}", "values": vec.tolist(), "metadata": meta}
           for row, vec, meta in zip(posts.to_dict("records"), vectors, metas)]
print(len(records), "records; the first:", records[0]["id"], f"with {len(records[0]['values'])} values")
print(records[0]["metadata"])

- 1,000 records. Each id is `post-` and the post's position at load, so a rerun writes the same ids; each holds 384 plain numbers and the metadata from 6.2.
- **Common mistake:** forgetting `.tolist()` on a Pinecone path. The `TypeError` comes before any request, which at least makes it quick to find.

**In an interview** ("Design the store for a company's documents"): keep the originals in files or a SQL table under a stable id, embed each with one model, and store the id, the vector and only the metadata you filter on and show. A search returns ids that lead back to the full document, and a rerun overwrites by id, so the two stores stay in step.

### 8.2 Checking before writing

`check_records(records, dim=384)` stops before anything is written if a record would do damage. `assert condition, message` stops the program with the message when the condition is false.

- **Ids unique:** a repeated id would make one post silently overwrite another.
- **Every vector 384 long:** a vector of another size came from another model, or a broken row.
- **No `None`:** Pinecone would refuse it with an HTTP 400 halfway through the write, and Chroma's `upsert` would drop the key without a word (6.1).

Two more rules were made safe earlier: no empty list (6.2 leaves the key out) and plain lists for Pinecone (8.1's `.tolist()`).

**Predict first:** what would a duplicate id do if it reached the database: raise an error, or something else?

In [ ]:
def check_records(records, dim=384):
    """Stop before writing if ids repeat, a vector has the wrong size or a metadata value is None."""
    ids = [r["id"] for r in records]
    assert len(ids) == len(set(ids)), "duplicate ids"
    for r in records:
        assert len(r["values"]) == dim, f"{r['id']} has {len(r['values'])} numbers, not {dim}"
        assert None not in r["metadata"].values(), f"{r['id']} has a None in its metadata"
    print(f"{len(records)} records OK")


check_records(records)                       # Try changing it: check_records(records + records[:1])

- `1000 records OK`.
- A duplicate id raises nothing in the database: the second record would silently overwrite the first, and one post would disappear. That is why the check runs first.
- Try changing it: call `check_records(records + records[:1])`, which adds a copy of the first record. It stops with `AssertionError: duplicate ids` and nothing is written. Put `check_records(records)` back and rerun.
- **Common mistake:** skipping the check because the upsert would fail anyway. A duplicate id does not fail; it overwrites.

**In an interview** ("What would you check before writing 1,000 records?"): that every id is unique, that every vector has the model's size, and that no metadata value is `None` or an empty list; checking first means nothing is half-written when one record is wrong.

### 8.3 Upsert in batches into Chroma

The cell opens a Chroma store on Colab's disk in a way that survives a rerun of the whole notebook:

- `chromadb.api.client.SharedSystemClient.clear_system_cache()` first. Chroma caches one client per folder inside the Python process; deleting the folder alone and creating `posts` again fails with `Collection [posts] already exists`.
- `shutil.rmtree("chroma_posts", ignore_errors=True)` then empties the folder, and `chromadb.PersistentClient(path="chroma_posts")` opens it.
- `create_collection("posts", embedding_function=None, configuration={"hnsw": {"space": "cosine"}})`: the cosine space is set explicitly because Chroma's default is l2, and with `embedding_function=None` Chroma never runs a model of its own. This is where Chroma is connected to the Hugging Face model: the vectors come from `embed`, and Chroma stores them as they are, without re-embedding.

Then one `upsert` call per batch of 100. **Upsert** inserts a new id and overwrites an existing one, so with stable ids a rerun replaces records instead of duplicating them.

**Predict first:** how many upsert calls will the loop make, and what will `count()` print?

In [ ]:
chromadb.api.client.SharedSystemClient.clear_system_cache()     # forget the cached client for the folder
shutil.rmtree("chroma_posts", ignore_errors=True)  # start from an empty store, so the notebook reruns from the top
client = chromadb.PersistentClient(path="chroma_posts")
posts_col = client.create_collection("posts", embedding_function=None,
                                     configuration={"hnsw": {"space": "cosine"}})

calls = 0
for i in range(0, len(records), UPSERT_BATCH):
    batch = records[i:i + UPSERT_BATCH]
    posts_col.upsert(ids=[r["id"] for r in batch],
                     embeddings=[r["values"] for r in batch],
                     metadatas=[r["metadata"] for r in batch])
    calls += 1
print(f"{calls} upsert calls; count() = {posts_col.count()}")

- Ten calls of 100 records, and `count()` prints 1000.
- If a batch had failed halfway, you would fix the cause and rerun: the stable ids mean the batches already written are overwritten, not duplicated.
- **Common mistake:** deleting the `clear_system_cache()` line. The first run still works, and the second run in the same session fails with `Collection [posts] already exists`.

**In an interview** ("Why upsert and not add?"): upsert inserts new ids and overwrites existing ones, so a rerun of the pipeline replaces the records instead of duplicating or rejecting them. Records go in batches so each request stays small and a failure costs one batch.

**Before you move on**, check that you can:

- build a record with an id, the values as a plain list, and metadata;
- write a check that stops on duplicate ids, a wrong vector size or a `None`;
- open a Chroma store so that a second run of the notebook starts clean;
- upsert records in batches and confirm the count.

## 9. Vector query with semantic search

### 9.1 A search function

`search(query, top_k=5, topic=None, org=None)` is a small retrieval application around the collection:

- It embeds the query with the same `embed` as the stored posts; `query` embeds nothing itself.
- `posts_col.query(query_embeddings=embed([query]), n_results=top_k, where=..., include=["distances", "metadatas"])` finds the nearest stored vectors, keeping only the records that pass `where` (built from `topic` and `org`, 9.2).
- The result is column-major: one list per field (`ids`, `distances`, `metadatas`), each holding one inner list per query sent, hence the `[0]` inside `chroma_hits`.

Chroma reports a **distance**: cosine distance = 1 − cosine similarity, so lower is closer and the first row is the best match.

**Predict first:** in each table below, will the best match be the row with the largest distance or the smallest?

In [ ]:
def search(query, top_k=5, topic=None, org=None):
    """The top_k posts nearest the query in Chroma, optionally only one topic and only posts whose orgs hold `org`."""
    filters = ([{"topic": topic}] if topic else []) + ([{"orgs": {"$contains": org}}] if org else [])
    where = {"$and": filters} if len(filters) == 2 else (filters[0] if filters else None)
    result = posts_col.query(query_embeddings=embed([query]), n_results=top_k,
                             where=where, include=["distances", "metadatas"])
    return chroma_hits(result)


for q in ["how far is it to Mars", "my brakes squeal when I stop", "is it safe to take aspirin every day"]:
    print(f"\n{q}")
    print(search(q))

- Each table is sorted nearest first, so the smallest distance is the best match: about 0.53 for the Mars query, 0.70 for the brakes and 0.67 for aspirin.
- With no filter at all, the Mars query's five rows are all `sci.space`, the aspirin query's all `sci.med`, and the brakes query's first four `rec.autos` (its fifth is a medical post). The topic came from meaning alone.
- Read a preview or two: the top rows are on the query's subject even where they share few words with it, such as posts about migraine drugs for the aspirin query.
- **Common mistake:** reading distance as similarity and taking the last row as the best match.

**In an interview** ("Walk me through a semantic search from a typed query to a ranked list"): the query is embedded with the same model and normalisation as the stored texts; the collection finds the nearest stored vectors by cosine distance, applies any metadata filter, and returns ids, distances and metadata, nearest first.

### 9.2 Filters

- **Topic:** `where={"topic": topic}` keeps only that topic's posts before ranking.
- **Organisation:** `where={"orgs": {"$contains": org}}` keeps posts whose `orgs` list holds the name. A post with no `orgs` key is not matched.
- **Both:** Chroma takes one condition as it is and two only inside `$and`: `{"$and": [{"topic": "sci.space"}, {"orgs": {"$contains": "NASA"}}]}`, which is what `search` builds.

The filter is applied inside the database, so Chroma ranks only the records that pass it. A very selective filter can still leave fewer than `top_k` matches.

**Predict first:** a search for space posts that mention NASA asks for five results. Will it get five?

In [ ]:
q = "orbit of a satellite around the moon"
print(search(q, topic="sci.space"))
print(search(q, topic="sci.space", org="NASA"))      # Try changing it: org="Toyota", topic="rec.autos"

- The first table holds space posts only, the nearest at about 0.41. The second holds only space posts whose `orgs` list holds NASA. Five come back, because 28 space posts carry NASA, plenty to rank; their distances are larger (about 0.66 to 0.81), because the filter removed the nearer posts that do not name NASA.
- Try changing it: set `org="Toyota", topic="rec.autos"` and rerun. Only the 10 car posts whose `orgs` holds Toyota are ranked, and the five nearest come back at distances of about 0.95 to 1.01: about as far as unrelated texts get, because none of them is about satellites (a cosine distance runs from 0 to 2). `org="VGA", topic="comp.graphics"` behaves the same way (11 posts, about 0.92 to 0.95). Put `org="NASA", topic="sci.space"` back.

The operator matters. `$contains` is Chroma's test for "this list holds the value"; `$in`, Pinecone's spelling, does something else in Chroma.

**Predict first:** how many records will `$in` against the `orgs` list match in Chroma?

In [ ]:
for where in [{"orgs": {"$contains": "NASA"}}, {"orgs": {"$in": ["NASA"]}}]:
    print(f"{len(posts_col.get(where=where, include=[])['ids']):4} records match {where}")

- `$contains` finds the 29 posts whose `orgs` holds NASA, across all topics (28 of them in `sci.space`); `$in` against the list field matches 0, and raises no error.
- **Common mistake:** using `$in` against the `orgs` list in Chroma. It matches nothing and raises no error, so the search just looks empty.

**In an interview** ("Integrate Chroma into a small search function with a topic filter"): embed the query with the same model, call `query` with `query_embeddings`, `n_results` and a `where` built from the filters (one condition as it is, two inside `$and`), and return ids, distances and metadata as a table.

### 9.3 Is the search any good?

A quick check set: eight queries whose right topic we know. For each, count how many of the top 5 come from that topic, and read the top distance. Topic is a stand-in for "relevant": a quick test, not the real one.

**Predict first:** of the eight check queries, how many will get all five results in the right topic?

In [ ]:
CHECKS = [("my rocket engine failed at launch", "sci.space"), ("side effects of a new medicine", "sci.med"),
          ("best oil for an old engine", "rec.autos"), ("rendering a 3D image file format", "comp.graphics"),
          ("orbit of a satellite around the moon", "sci.space"), ("my doctor says I have a migraine", "sci.med"),
          ("dealer price for a new sedan", "rec.autos"), ("convert a GIF to JPEG", "comp.graphics")]
for q, topic in CHECKS:
    hits = search(q)
    print(f"{(hits['topic'] == topic).sum()}/5  top distance {hits['distance'].iloc[0]:.2f}  {q}  ({topic})")

- Seven queries put all five results in the right topic and "convert a GIF to JPEG" puts four of five. Top distances run from about 0.36 to 0.56.

Now read the one miss whole: an off-topic row is often a post about the query's meaning that was filed under another topic.

**Predict first:** the GIF query's off-topic row: will it be a post unrelated to images, or one about images posted in another group?

In [ ]:
print(search("convert a GIF to JPEG"))

- The off-topic row is the third, a `sci.space` post announcing "OZONE GIF IMAGES", at about 0.52, among the on-topic distances. It is about GIF images, posted in a space group: by meaning it is a fair answer, and the topic label calls it a miss. Whether it is a poor answer is a judgement the label cannot make.
- A team builds a **relevance set** next: test queries, each paired with the ids a person judged correct. **recall@k** is the share of the correct ids that appear in the top k; **MRR** (mean reciprocal rank) rewards a correct hit near the top. Sentence Transformers' `InformationRetrievalEvaluator` scores a search this way.
- **Common mistake:** treating the topic as proof of relevance. A post in the right topic can still be a poor answer, and an off-topic post can be a good one.

**In an interview** ("How do you know the search is any good?"): run queries whose answers you know and count how many come back right. Topic labels give a quick check; a labelled relevance set scored with recall@k or MRR is the real test.

### 9.4 A query with no good answer

"what is a good recipe for banana bread": nothing in four groups about space, medicine, cars and graphics answers it.

**Predict first:** will `search` return fewer than five rows for it?

In [ ]:
hits = search("what is a good recipe for banana bread")
print(hits)
CUTOFF = 0.65                                    # read from this data: see the bullets below
print(f"\nrows under the cut-off {CUTOFF}: {(hits['distance'] < CUTOFF).sum()}")

- Five rows still come back, at distances of about 0.71 to 0.80, well above the check queries' top distances of about 0.36 to 0.56. A nearest-neighbour search always returns the nearest `top_k`, however far they are.
- The nearest are posts about food, such as the food astronauts eat: the model found the closest meaning there is, and nothing here answers the question.
- A cut-off near 0.65 separates the two sets on this data, so a search function could say "no good match" instead of returning five poor rows. Here no row is under it.
- **Common mistake:** using 0.65 as a rule everywhere. It was read from these distances, with this model on this data. Another model or dataset needs its own, chosen the same way: run queries that have answers and queries that do not, and pick a value between the two sets of distances.

**In an interview:** a vector search never says "nothing found" by itself; it returns the nearest `top_k`. Deciding that nothing is good enough is the application's job, with a distance cut-off read from queries with and without answers.

**Before you move on**, check that you can:

- write a search function that embeds the query with the same model and returns a table nearest first;
- filter by a string field and by a list field in Chroma, and combine two filters;
- say which operator matches a list field in Chroma and which matches nothing;
- check a search with a few queries whose answers you know, and pick a cut-off for "no good match".

## 10. The same in Pinecone

The same 1,000 records go into your own Pinecone index. Everything above runs without a key; from here on each cell talks to Pinecone's servers.

### 10.1 Upsert in batches

First the key. It lives in Colab's Secrets panel as `PINECONE_API_KEY` with "Notebook access" switched on, and the only line that reads it is `pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))`. No cell prints the key or writes it anywhere, so a saved copy of this notebook never holds it. Outside Colab the cell reads an environment variable of the same name instead.

If the cell stops with `SecretNotFoundError`, Colab found no secret of that name with notebook access on: open the key icon on the left, add `PINECONE_API_KEY` with your key as its value, switch on notebook access, and rerun the cell.

**Predict first:** this cell lists your indexes. Should `posts-thursday` be among them before the next cell runs?

In [ ]:
try:
    from google.colab import userdata          # Colab: the key comes from the Secrets panel
    pc = Pinecone(api_key=userdata.get("PINECONE_API_KEY"))
except ImportError:                           # outside Colab: an environment variable of the same name
    pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])
print(f"connected; posts-thursday exists already: {pc.has_index('posts-thursday')}")

- `False` on a clean project: the last run's 10.3 deleted it. `True` means an earlier run stopped before 10.3; the next cell then skips the create, and the upsert overwrites the same ids.

The index: `dimension=384` is MiniLM's output size and `metric="cosine"` matches the normalised vectors. The free Starter plan's indexes all live on AWS `us-east-1`, and the plan allows 5 serverless indexes per project; this notebook uses one. Then `upsert(vectors=..., namespace="posts")` in batches of 100. On the pinned 9.1.0 `upsert` and `query` take keyword arguments only. Pinecone takes at most 1,000 records and 2 MB per upsert request, so batches of 100 stay well inside.

Pinecone is **eventually consistent**: there can be a short delay before new data is visible, so a count read straight after an upsert may lag. `wait_for_count(index, 1000)` from the setup cell polls every 2 seconds for up to 60 seconds and prints the count.

**Predict first:** if you read the count one second after the last upsert, will it say 1,000?

In [ ]:
if not pc.has_index("posts-thursday"):
    pc.create_index(name="posts-thursday", dimension=384, metric="cosine",
                    spec=ServerlessSpec(cloud="aws", region="us-east-1"))
index = pc.Index("posts-thursday")

start = time.perf_counter()
for i in range(0, len(records), UPSERT_BATCH):
    index.upsert(vectors=records[i:i + UPSERT_BATCH], namespace="posts")     # the same records Chroma stored
print(f"{len(range(0, len(records), UPSERT_BATCH))} upsert calls in {time.perf_counter() - start:.1f} s; "
      f"count right away: {index.describe_index_stats().total_vector_count}")
n_vectors = wait_for_count(index, 1000)

- Ten upsert calls, and `wait_for_count` prints `1000 vectors` once Pinecone has caught up. On our run the count already read 1,000 straight after the upsert; it does not always, which is why the wait is there.
- All 1,000 posts sit in one namespace, `posts`, so the total count is the namespace's count.
- **Common mistake:** a numpy array as `values`. On 9.1.0 it raises `TypeError: Type is not JSON serializable: numpy.float32` before any request; the records already hold `vec.tolist()`.
- **Common mistake:** a key on screen or in a cell. Keep the Secrets panel closed while sharing your screen, and if a key ever leaks, revoke it in the Pinecone console and make a new one.

**In an interview** ("Upload embeddings to Pinecone"): create a serverless index with the model's dimension and the cosine metric, upsert records of id, values as a plain list and metadata in batches well under the request limits, and wait for the count, because Pinecone is eventually consistent.

### 10.2 Filtered queries

`index.query(vector=..., top_k=5, namespace="posts", filter=..., include_metadata=True)` takes the query vector as a plain list from the same `embed`. Each of `response.matches` has `match.id`, `match.score` and `match.metadata`, which is `None` unless `include_metadata=True`. `pinecone_hits` turns them into a table.

With `metric="cosine"`, Pinecone reports a **score**, the cosine similarity, so higher is closer. Chroma reported a distance, 1 − similarity.

**Predict first:** for the same query and the same vectors, Chroma gave the top space post a distance of d. What score will Pinecone give it?

In [ ]:
q = "orbit of a satellite around the moon"
q_vector = embed([q])[0].tolist()              # a plain list: the pinned client cannot send a numpy array
for _ in range(15):                            # a query can lag a moment behind the count: re-read until full
    by_topic = index.query(vector=q_vector, top_k=5, namespace="posts",
                           filter={"topic": {"$eq": "sci.space"}}, include_metadata=True)
    if len(by_topic.matches) == 5:
        break
    time.sleep(2)
print(pinecone_hits(by_topic))

both = pinecone_hits(by_topic).set_index("id")[["score"]].join(search(q, topic="sci.space").set_index("id")["distance"])
both["score + distance"] = (both["score"] + both["distance"]).round(3)
print(both)

- The same five space posts as Chroma's topic-filtered search in 9.2, in the same order, best first, now with the highest score on top.
- The loop re-reads the query until five matches come back: even after the count reads 1,000, a query can lag a moment behind it. On our run the first read was already complete.
- Score plus distance is 1.0 for every row: with the same vectors, score = 1 − distance, so a distance of 0.36 is a score of about 0.64.
- **Common mistake:** reading Pinecone's score as a distance, so the best match is read from the wrong end of the list.

Now the list field. `{"orgs": {"$in": ["NASA"]}}` matches a post when any value in its `orgs` list is in `["NASA"]`. It is the same question Chroma answered with `$contains` in 9.2, in Pinecone's spelling.

**Predict first:** this filter has no topic in it. Will every post it returns be from `sci.space`?

In [ ]:
by_org = index.query(vector=q_vector, top_k=5, namespace="posts",
                     filter={"orgs": {"$in": ["NASA"]}}, include_metadata=True)
print(pinecone_hits(by_org))

- Every row's `orgs` holds NASA, and these are the same five posts, in the same order, as Chroma's `topic="sci.space", org="NASA"` search in 9.2. Here all five are space posts, but the filter asks only about `orgs`, so a post from another topic that names NASA could appear; add a `topic` condition to keep one topic only.
- A bare `{"orgs": "NASA"}` matches the same posts; an array as the bare value is an HTTP 400 error. `top_k` can go up to 10,000 and `$in` holds up to 10,000 values.

**In an interview** ("Return ranked results from Pinecone with a metadata filter: what do the scores mean?"): query with the vector from the same model as a plain list, `top_k`, the namespace and a filter such as `{"topic": {"$eq": ...}}` or `{"orgs": {"$in": [...]}}`; a list field matches when any of its values passes. With the cosine metric the score is the cosine similarity, higher is closer, and equals 1 minus Chroma's distance for the same vectors.

### 10.3 Clean up

Every notebook this week ends its Pinecone part by deleting the day's index, so your free project is empty between days and a rerun starts clean. The guard is the same one the create cell uses, so running this cell twice raises no error.

**Predict first:** what happens if you run this cell a second time?

In [ ]:
if pc.has_index("posts-thursday"):
    pc.delete_index("posts-thursday")
print(f"posts-thursday still there: {pc.has_index('posts-thursday')}")

- `False`: the index is gone. A second run finds no index, deletes nothing and raises no error.
- The Chroma folder `chroma_posts` stays on Colab's disk until the runtime is recycled; the next run's 8.3 empties it first.
- **Common mistake:** skipping the clean-up when time is short. An index left behind keeps counting against your free project.

**In an interview:** delete what you created when you are done, behind the same existence check the create used, so a free-tier project stays empty and a rerun never fails on a missing index.

**Before you move on**, check that you can:

- read a Pinecone key from Colab Secrets in one line, and say what `SecretNotFoundError` means;
- create an index behind `has_index`, upsert in batches into a namespace and wait for the count;
- query with a topic filter and with a list filter, and convert a Pinecone score to a Chroma distance;
- delete the index behind the same guard.

## Recap

| Idea | What it is for | Watch out for |
|---|---|---|
| Semantic against keyword search | finding paraphrases (meaning) and exact strings (keywords) | keyword search wins on codes and names; teams run both (hybrid search) |
| Embedding model | turning each text into a fixed-length vector the database compares | the same model and normalisation at upsert and query; a same-size swap fails silently |
| Data preparation | a stable id per document and a sample small enough to rerun | topic names come from `target_names`, which is sorted |
| Cleaning and length | text worth returning, of a known length | clean before judging length; the model silently reads only 256 word pieces |
| NER | entities as metadata a user filters on | forum text gets wrong labels; read the top entities before filtering on them |
| Metadata | fields to filter on and fields to show | no `None`; no empty list in Chroma, so leave the key out |
| Batches | bounded memory, progress, and a failure costing one batch | batch size changes speed and memory, not the vectors |
| Records and upsert | id, plain-list values and metadata, checked before writing | `.tolist()` for Pinecone; stable ids make a rerun an overwrite |
| Search and filters | ranked results, narrowed by topic or organisation | Chroma's distance is lower-is-closer, Pinecone's score higher-is-closer; `$contains` in Chroma, `$in` in Pinecone; a nonsense query still returns rows |

**Quick check**
1. How does semantic search differ from keyword search, and when does each find the better result?
2. What does an embedding model do in a vector database, and what goes wrong if the queries use a different model?
3. What does named entity recognition find, and how would you use it in a search index?
4. Why does `make_metadata` leave out a key whose list is empty?
5. Why embed 1,000 posts in batches, and does changing the batch size change the vectors?
6. For one post Chroma reports a distance of 0.36. What score does Pinecone give the same vector, and which end of each list is the best match?

<details>
<summary>Answers (say yours aloud first)</summary>

1. Keyword search ranks by shared words, so it scored 0 for the engine sentence against "my car will not start when it is freezing". Semantic search compares embeddings from a model trained on paraphrases, and it found that sentence first. Keyword search wins on exact strings such as part numbers, where MiniLM ranked 7713-B above 7731-B; hybrid search runs both.
2. It produces every vector; the database only stores and compares them. A query model with a different size fails loudly on the dimension. A different model of the same size returns confident wrong answers with no error, so one model, with the same normalisation, serves upsert and query, and a new model means re-embedding everything.
3. Named spans such as organisations, people and places, each labelled, like NASA as ORG and Houston as GPE. The entities become metadata lists a user can filter on, after a look at the top entities per group, because on forum text the model tags things like a signature e-mail address or MSG as organisations.
4. Chroma 1.5.9 refuses an empty list with a `ValueError` (Pinecone accepted one in our test, but one refusal is enough), and `None` is refused by Pinecone and by Chroma's `add`, and silently dropped by Chroma's `upsert`. 359 of the 1,000 posts name no organisation, so writing `"orgs": []` would fail the upsert; a missing key simply does not match an organisation filter.
5. Batches keep memory flat, show progress and limit a failure to one batch. The batch size changes speed and memory only; the vectors agree to within 0.00001, and the rankings do not change.
6. About 0.64, since score = 1 − distance for cosine. Chroma's best match is the row with the lowest distance; Pinecone's is the row with the highest score.

</details>